# Readiness trace

Collects everything needed to test two claims about masked diffusion decoding on a
pretrained model. 

It measures:

- `gt_margin(i) = p(x_i) − max_{v ≠ x_i} p(v)` (how predictable a masked
  position is). Teacher-forced: the true token only indexes the distribution.
- `wait_gain(i) = gt_margin_after − gt_margin_before` (how much position `i`
  gains from waiting, after half the other masked positions are revealed with
  their true tokens).

The question is whether the decoder's own confidence tracks the first and not the
second, and whether the residual stream, in particular the final representation
feeding the unembedding, carries the second.

In the readiness file, every feature and every hidden state comes from the pass before the reveal. Only the target uses the second pass. There is a check at the end that catches this.

In [ ]:
# config
MODEL_ID  = "GSAI-ML/LLaDA-8B-Base"     # or "Dream-org/Dream-v0-Base-7B"
MASK_ID   = None      # None = auto-detect. LLaDA-8B is 126336.
DTYPE     = "bfloat16"

# Held-out text. This should be as close as possible to the model's pretraining
# distribution: feeding it a foreign corpus measures its surprise at that corpus,
# not readiness. Replace with your own list of strings if you have something better.
CORPUS    = ("wikitext", "wikitext-103-raw-v1", "validation")

N_WINDOWS = 200       # 100 halves the storage and is an acceptable fallback
SEQ_LEN   = 256
LEVELS    = (0.2, 0.5, 0.8)
N_BLOCKS  = 3         # transformer blocks sampled across depth, PLUS the final
                      # normalised representation (always included)
BATCH     = 4
LOGIT_CHUNK = 1024    # rows per softmax chunk; lower it if you hit OOM
SEED      = 0
OUT_DIR   = "traces"

In [ ]:
# !pip install -q transformers datasets scikit-learn
import os, math, json
from types import SimpleNamespace
import numpy as np
import torch

torch.set_grad_enabled(False)
os.makedirs(OUT_DIR, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| torch", torch.__version__)

In [ ]:
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
try:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, trust_remote_code=True, torch_dtype=getattr(torch, DTYPE))
except Exception as exc:
    print("AutoModelForCausalLM failed, trying AutoModel:", exc)
    model = AutoModel.from_pretrained(
        MODEL_ID, trust_remote_code=True, torch_dtype=getattr(torch, DTYPE))
model = model.to(DEVICE).eval()
print(type(model).__name__)

## Adapter check

Three things are settled here: the mask token, which tensor is the final normalised representation, and which layers to store.

The final representation is identified by reconstructing the logits from it.
If `head(h_last) == logits`, then a linear probe on `h_last` spans every linear
readout of the logits, which is the whole point of storing it. If the check
fails, the model does not expose hidden states in the standard way and the fix is
a forward hook on whatever module feeds the head.

In [ ]:
# mask token 
if MASK_ID is None:
    MASK_ID = getattr(tokenizer, "mask_token_id", None)
if MASK_ID is None and "LLaDA" in MODEL_ID:
    MASK_ID = 126336                      # documented in the LLaDA repo
assert MASK_ID is not None,
print("mask token id:", MASK_ID)

# one probe forward 
vsz = getattr(tokenizer, "vocab_size", None) or getattr(model.config, "vocab_size")
probe = torch.randint(0, min(1000, vsz), (1, 32), device=DEVICE)
out = model(probe, output_hidden_states=True)
assert getattr(out, "hidden_states", None) is not None, \
    "this model does not return hidden_states; use forward hooks instead"

hs = out.hidden_states
N_HIDDEN, D = len(hs), hs[-1].shape[-1]
print(f"{N_HIDDEN} hidden tensors (embeddings + blocks), d_model = {D}")

# is hs[-1] the representation the head reads?
head = model.get_output_embeddings()
if head is None:
    head = getattr(model, "lm_head", None)
assert head is not None, 

recon = head(hs[-1][:, :8].float() if hs[-1].dtype != torch.float32 else hs[-1][:, :8])
ref = out.logits[:, :8].float()
diff = (recon.float() - ref).abs().max().item()
scale = ref.abs().max().item()
print(f"max |head(h_last) - logits| = {diff:.4g}   (logit scale {scale:.4g})")
if diff < 0.05 * scale:
    print("OK - hs[-1] is the final normalised representation feeding the head.")
else:
    print("MISMATCH - hs[-1] is not what the head reads. Do not trust the last layer;")
    print("register a forward hook on the module that feeds the head and store that.")

In [ ]:
# which layers to store 
# N_BLOCKS blocks spread across depth, plus the final representation.
n_blocks = N_HIDDEN - 1                     # index 0 is the embedding output
picks = [int(round(n_blocks * f)) for f in
         [(i + 1) / (N_BLOCKS + 1) for i in range(N_BLOCKS)]]
LAYER_IDX = sorted(set(picks + [N_HIDDEN - 1]))
print("storing hidden tensors at indices:", LAYER_IDX,
      f"(last = {N_HIDDEN - 1} is the final representation)")

est = lambda rows: rows * D * 2 * len(LAYER_IDX) / 1e9
approx = N_WINDOWS * SEQ_LEN * sum(LEVELS)
print(f"estimated size: predictability ~{est(approx):.1f} GB, "
      f"readiness ~{est(approx / 2):.1f} GB")

### Fallback if the check above failed

Some `trust_remote_code` models do not return `hidden_states`. This wraps the
model with forward hooks so the rest of the notebook is unchanged: list the
modules you want captured, **last one being whatever feeds the head**, run the
cell, and carry on.

In [ ]:
# Uncomment and adapt if the adapter check failed.


# for name, mod in model.named_modules():
#     if isinstance(mod, torch.nn.ModuleList) and len(mod) > 4:
#         print(f"{name:55} ModuleList[{len(mod)}]")
# for name, mod in model.named_modules():
#     if name.endswith(("norm", "ln_f", "final_layer_norm")):
#         print(f"{name:55} {type(mod).__name__}")

class HookedModel(torch.nn.Module):
    """Makes any model look like one that supports output_hidden_states."""
    def __init__(self, model, modules):
        super().__init__()
        self.model, self.mods = model, list(modules)
    def forward(self, x, output_hidden_states=True):
        store, handles = {}, []
        for k, m in enumerate(self.mods):
            handles.append(m.register_forward_hook(
                lambda mod, inp, out, k=k: store.__setitem__(
                    k, out[0] if isinstance(out, tuple) else out)))
        try:
            out = self.model(x)
        finally:
            for h in handles:
                h.remove()
        return SimpleNamespace(logits=out.logits,
                               hidden_states=tuple(store[k] for k in range(len(self.mods))))

# blocks = model.model.transformer.blocks      # <- adapt these two lines
# final  = model.model.transformer.ln_f
# picked = [blocks[len(blocks) // 4], blocks[len(blocks) // 2],
#           blocks[3 * len(blocks) // 4], final]
# model = HookedModel(model, picked)
# LAYER_IDX = list(range(len(picked)))
# print("hooked; LAYER_IDX =", LAYER_IDX, "(last is the final representation)")

In [ ]:
# corpus -> windows 
from datasets import load_dataset

ds = load_dataset(CORPUS[0], CORPUS[1], split=CORPUS[2])
text = "\n\n".join(t for t in ds["text"] if t.strip())
ids = tokenizer(text, add_special_tokens=False, return_tensors="pt").input_ids[0]
need = N_WINDOWS * SEQ_LEN
assert len(ids) >= need, f"corpus too short: {len(ids)} tokens, need {need}"
windows = ids[:need].view(N_WINDOWS, SEQ_LEN).contiguous()
assert (windows == MASK_ID).sum() == 0, "the corpus contains the mask token"
print("windows:", tuple(windows.shape))
print("sample:", repr(tokenizer.decode(windows[0][:40])))

In [ ]:
# helpers 
@torch.no_grad()
def position_stats(logits, ids, chunk=1024):
    """logits [n, V] at selected positions, ids [n] the true tokens there."""
    keys = ("confidence", "entropy", "margin", "gt_margin")
    out = {k: [] for k in keys}
    for s in range(0, logits.shape[0], chunk):
        lg = logits[s:s + chunk].float()
        ii = ids[s:s + chunk]
        logp = lg.log_softmax(-1)
        p = logp.exp()
        ent = -(p * logp).sum(-1)
        v, idx = p.topk(2, dim=-1)
        conf, second = v[:, 0], v[:, 1]
        p_gt = p.gather(1, ii[:, None]).squeeze(1)
        best_other = torch.where(idx[:, 0] == ii, second, conf)
        out["confidence"].append(conf.cpu())
        out["entropy"].append(ent.cpu())
        out["margin"].append((conf - second).cpu())
        out["gt_margin"].append((p_gt - best_other).cpu())
    return {k: torch.cat(v) for k, v in out.items()}


def make_masks(ids, t, generator):
    """-> masked [B, L] bool, and still [B, L]: the half NOT revealed in pass B."""
    B, L = ids.shape
    masked = torch.rand(B, L, generator=generator) < t
    for b in range(B):
        if masked[b].sum() < 2:
            masked[b, torch.randperm(L, generator=generator)[:2]] = True
    still = torch.zeros_like(masked)
    for b in range(B):
        idx = masked[b].nonzero().squeeze(1)
        perm = idx[torch.randperm(len(idx), generator=generator)]
        still[b, perm[len(perm) // 2:]] = True
    return masked, still


def reveal(ids, masked, still, mask_id):
    xa = ids.clone(); xa[masked] = mask_id      # pass A: everything masked
    xb = ids.clone(); xb[still] = mask_id       # pass B: half revealed, true tokens
    return xa, xb

In [ ]:
@torch.no_grad()
def collect(model, windows, mask_id, levels, layer_idx, batch, seed, chunk):
    gt = {k: [] for k in ("gt_margin", "confidence", "entropy", "margin",
                          "t", "window", "position")}
    wg = {k: [] for k in ("wait_gain", "confidence", "entropy", "margin",
                          "t", "window", "position")}
    gt_h = {l: [] for l in layer_idx}
    wg_h = {l: [] for l in layer_idx}
    g = torch.Generator().manual_seed(seed)

    for t in levels:
        for s in range(0, len(windows), batch):
            ids = windows[s:s + batch]
            B, L = ids.shape
            wid = torch.arange(s, s + B)[:, None].expand(B, L)
            pos = torch.arange(L)[None, :].expand(B, L)

            masked, still = make_masks(ids, t, g)
            xa, xb = reveal(ids, masked, still, mask_id)

            # pass A: features, hidden states, and gt_margin "before"
            oa = model(xa.to(DEVICE), output_hidden_states=True)
            sel = masked.reshape(-1)
            dev = oa.logits.device
            la = oa.logits.reshape(-1, oa.logits.shape[-1])[sel.to(dev)]
            st = position_stats(la, ids.reshape(-1)[sel].to(dev), chunk)

            for k in ("gt_margin", "confidence", "entropy", "margin"):
                gt[k].append(st[k])
            gt["t"].append(torch.full((int(sel.sum()),), float(t)))
            gt["window"].append(wid.reshape(-1)[sel])
            gt["position"].append(pos.reshape(-1)[sel])
            for l in layer_idx:
                h = oa.hidden_states[l]
                gt_h[l].append(h.reshape(-1, h.shape[-1])[sel.to(h.device)].half().cpu())
            del oa, la

            # pass B: gt_margin "after", at the positions still masked
            sub = still.reshape(-1)[sel]           # index into the pass-A rows
            ob = model(xb.to(DEVICE), output_hidden_states=True)
            selb = still.reshape(-1)
            lb = ob.logits.reshape(-1, ob.logits.shape[-1])[selb.to(ob.logits.device)]
            stb = position_stats(lb, ids.reshape(-1)[selb].to(lb.device), chunk)
            del ob, lb

            # features and hidden states from pass A; only the target from pass B
            wg["wait_gain"].append(stb["gt_margin"] - st["gt_margin"][sub])
            for k in ("confidence", "entropy", "margin"):
                wg[k].append(st[k][sub])
            wg["t"].append(torch.full((int(selb.sum()),), float(t)))
            wg["window"].append(wid.reshape(-1)[selb])
            wg["position"].append(pos.reshape(-1)[selb])
            for l in layer_idx:
                wg_h[l].append(gt_h[l][-1][sub])

        print(f"  level t={t} done", flush=True)

    out_gt = {k: torch.cat(v) for k, v in gt.items()}
    out_wg = {k: torch.cat(v) for k, v in wg.items()}
    out_gt["hidden"] = {l: torch.cat(v) for l, v in gt_h.items()}
    out_wg["hidden"] = {l: torch.cat(v) for l, v in wg_h.items()}
    return out_gt, out_wg

In [ ]:
import time
t0 = time.time()
rec_gt, rec_wg = collect(model, windows, MASK_ID, LEVELS, LAYER_IDX,
                         BATCH, SEED, LOGIT_CHUNK)
print(f"\n{time.time() - t0:.0f}s")
print("predictability rows:", len(rec_gt["gt_margin"]),
      "| readiness rows:", len(rec_wg["wait_gain"]),
      "| windows:", len(torch.unique(rec_gt["window"])))

In [ ]:
meta = dict(model=MODEL_ID, mask_id=int(MASK_ID), seq_len=SEQ_LEN,
            n_windows=N_WINDOWS, levels=list(LEVELS), layer_idx=list(LAYER_IDX),
            n_hidden_tensors=N_HIDDEN, d_model=int(D), seed=SEED,
            corpus=list(CORPUS), dtype=DTYPE)
rec_gt["meta"] = meta; rec_wg["meta"] = meta

torch.save(rec_gt, f"{OUT_DIR}/labels_gtmargin.pt")
torch.save(rec_wg, f"{OUT_DIR}/labels_waitgain.pt")
with open(f"{OUT_DIR}/meta.json", "w") as fh:
    json.dump(meta, fh, indent=2)

for f in ("labels_gtmargin.pt", "labels_waitgain.pt"):
    print(f, f"{os.path.getsize(f'{OUT_DIR}/{f}') / 1e9:.2f} GB")

## Sanity checks

The reference column is a 30M model trained on TinyStories. These are reference
points, not expected values, if the numbers differ, that is the result. What
would indicate a wiring problem is a sign flip or an order-of-magnitude gap,
above all on the first row.

If `corr(confidence, wait_gain)` comes out strongly positive, the two passes
have been swapped and the readiness features are being read after the reveal.

In [ ]:
def corr(a, b):
    return float(torch.corrcoef(torch.stack([a.float(), b.float()]))[0, 1])

wgv, conf = rec_wg["wait_gain"], rec_wg["confidence"]
rows = [
    ("corr(confidence, wait_gain)", corr(conf, wgv),            "-0.18"),
    ("share wait_gain > 0",          float((wgv > 0).float().mean()), "0.69"),
    ("mean wait_gain",               float(wgv.mean()),          "0.23"),
    ("mean gt_margin",               float(rec_gt["gt_margin"].mean()), "-"),
    ("mean confidence",              float(rec_gt["confidence"].mean()), "0.46"),
]
print(f"{'':34}{'this model':>12}{'30M ref':>12}")
print("-" * 58)
for name, got, ref in rows:
    print(f"{name:34}{got:>12.4f}{ref:>12}")

# provenance check: readiness features must come from pass A
key = {(int(w), int(p), round(float(t), 3)): i for i, (w, p, t)
       in enumerate(zip(rec_gt["window"], rec_gt["position"], rec_gt["t"]))}
bad = sum(abs(float(rec_wg["confidence"][i])
              - float(rec_gt["confidence"][key[(int(rec_wg["window"][i]),
                                                int(rec_wg["position"][i]),
                                                round(float(rec_wg["t"][i]), 3))]])) > 1e-5
          for i in range(0, len(wgv), max(1, len(wgv) // 500)))
print(f"\nfeature provenance: {bad} mismatches out of ~500 sampled rows (0 expected)")

## The headline number

One grouped split, one ridge, out-of-sample R².

The output baseline should explain predictability well and
readiness badly, and adding the residual stream should help readiness much
more than it helps predictability.

For reference, the 30M model gives 0.59 / 0.09 for the baseline, and increments
of +0.012 on predictability against +0.035 on readiness.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

ALPHAS = (1.0, 10.0, 100.0, 1e3, 1e4, 1e5)

def quick_r2(rec, target, dims=50, seed=0):
    """Grouped 50/25/25 split by window; alpha on val; refit on train+val."""
    y = rec[target].float().numpy()
    w = rec["window"].numpy()
    ids = np.unique(w)
    perm = np.random.default_rng(seed).permutation(ids)
    n = int(round(len(ids) * 0.25))
    te = np.isin(w, perm[:n]); va = np.isin(w, perm[n:2 * n]); tr = ~(te | va)

    base = np.stack([rec[k].float().numpy()
                     for k in ("confidence", "entropy", "margin", "t")], 1)
    blocks = []
    for l in sorted(rec["hidden"]):
        h = rec["hidden"][l].float().numpy()
        k = min(dims, h.shape[1], int(tr.sum()))
        blocks.append(PCA(n_components=k, random_state=0).fit(h[tr]).transform(h))
    full = np.concatenate([base] + blocks, 1)

    def score(x):
        sc = StandardScaler().fit(x[tr])
        xt, xv = sc.transform(x[tr]), sc.transform(x[va])
        a = max(ALPHAS, key=lambda a: Ridge(alpha=a).fit(xt, y[tr]).score(xv, y[va]))
        f = tr | va
        sc = StandardScaler().fit(x[f])
        return Ridge(alpha=a).fit(sc.transform(x[f]), y[f]).score(sc.transform(x[te]), y[te])

    return score(base), score(full)

print(f"{'target':18}{'output + t':>12}{'+ residual':>12}{'increment':>12}")
print("-" * 54)
for rec, tgt, label in ((rec_gt, "gt_margin", "predictability"),
                        (rec_wg, "wait_gain", "readiness")):
    b, f = quick_r2(rec, tgt)
    print(f"{label:18}{b:>12.4f}{f:>12.4f}{f - b:>+12.4f}")

## What to send back

`traces/labels_gtmargin.pt`, `traces/labels_waitgain.pt`, `traces/meta.json`,
and the printed output of the last three cells.

The full analysis on the other side adds what this notebook deliberately leaves
out: a dimension ladder including an uncompressed rung, five grouped splits, and
generation-clustered bootstrap intervals with the two targets differenced inside
each draw.

**Not covered here.** The commit-dependence horizon, whether the dependence
between positions committed together is decodable several steps before the
commit. That one needs real decoding plus k sequential forward passes per probed
step in two reveal orders, which is roughly two orders of magnitude more compute.
Worth doing only if the above comes back interesting.